# Batch culture: 25 species, one starting resource, two cross-fed metabolites

Run all cells from this project folder. Parameters are normalized exploratory values, not measured bacterial rates. See README.md for all equations and assumptions. The notebook uses the same scripts as the command-line simulation.

In [2]:
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
from IPython.display import display, Image
from model import Config, generate, simulate, summarize, export
from plots import make_plots, ensemble, ensemble_plot
config = Config(seed=42)
out = Path("outputs")
config

ModuleNotFoundError: No module named 'model'

## Generate and characterize the community
Growth-capacity scores and death rates are independently sampled truncated Gaussians. Food subsets and secretion partitions are sampled once, not at each time step. Metabolite-only consumers begin without food. Maximum growth is normalized across different uptake profiles.

In [ ]:
community = generate(config)
display(community["table"])
columns=["growth_score", "mu_max", "death_rate", "total_uptake_capacity", "M1_secretion_share", "M2_secretion_share"]
spread=community["table"][columns].describe().T
spread["median"]=community["table"][columns].median()
out.mkdir(exist_ok=True)
spread.to_csv(out / "realized_parameter_spread.csv")
display(spread)
permissions=community["table"][["can_consume_R0", "can_consume_M1", "can_consume_M2"]].value_counts().rename("species_count")
display(permissions)

## Integrate the batch
The starting nutrient is finite. Only consumption of R0 generates M1/M2. No chemical decays or replenishes. Constant death removes biomass to an accounting sink. Threshold crossings irreversibly remove species.

In [ ]:
result=simulate(community)
summary=summarize(community,result)
export(community,result,out)
figures=make_plots(community,result,out)
print("Survivors:", int(summary.survived.sum()), "/", config.n_species)
print("Maximum budget error:", abs(result["budget_error"]).max())
display(summary[["species", "mu_max", "death_rate", "peak_biomass", "final_biomass", "final_percent", "extinction_time", "survived"]])

## Species traits and batch dynamics
Colors consistently identify species in trajectory panels. The logarithmic biomass display uses a floor for extinct species. Composition is a percentage of current living biomass; increasing percentage can accompany decreasing absolute biomass.

In [ ]:
for path in figures[:2]:
    display(Image(filename=str(path)))

## Growth, death, trait associations and diversity
Gross growth depends on food availability. Net growth subtracts constant death. Biomass death flux changes because living biomass changes. Trait–outcome scatter plots describe this community; they do not isolate causal effects.

In [ ]:
for path in figures[2:]:
    display(Image(filename=str(path)))

## Extinction-threshold sensitivity
All random draws are held fixed. The thresholds are numerical conventions rather than universal biological detection limits.

In [ ]:
sensitivity=[]
for fraction in [1e-4, 1e-6, 1e-8]:
    pp=generate(replace(config, extinction_fraction=fraction))
    rr=simulate(pp)
    sensitivity.append({"cutoff_fraction":fraction,"survivors":int((rr["N"][-1]>0).sum()),"final_total_biomass":rr["N"][-1].sum(),"max_budget_error":abs(rr["budget_error"]).max()})
sensitivity=pd.DataFrame(sensitivity)
sensitivity.to_csv(out/"cutoff_sensitivity.csv",index=False)
display(sensitivity)

## Repeat over 20 random communities
Species within each community interact. These pooled correlations are descriptive, and the 500 rows are not 500 independent experiments. Biomass-time integral captures performance even if all species remain above the extinction cutoff.

In [ ]:
repeated=ensemble(config,replicates=20)
repeated.to_csv(out/"ensemble_species_summary.csv",index=False)
display(repeated.groupby("community_seed").agg(survivors=("survived","sum"),final_biomass=("final_biomass","sum")))
display(Image(filename=str(ensemble_plot(repeated,out))))

## Interpretation and next experiments
Compare absolute biomass, shares, and growth rates together. Final survival depends on observation time, death scale and cutoff; it does not prove permanent coexistence. Later experiments can compare no leakage versus leakage, altered death scales, or an imposed growth-survival tradeoff using the same community draws. A chemical conversion/necromass model would be a substantive extension, not just a parameter change.